# syft-ingest API Exploration

In [ ]:
import syft_ingest as si

## 1. Simplified gather() API — YouTube video

In [ ]:
# Simplest case: just platform + URLs
corpus = si.gather(
    "youtube", ["https://www.youtube.com/watch?v=zY2dAK-pMPI&t=11s"]
)  # Andrew Trask: talk

print(f"Items fetched: {len(corpus.all_items())}")
if corpus.all_items():
    item = corpus.all_items()[0]
    print(f"  Title: {item.title}")
    print(f"  Author: {item.author}")

In [ ]:
corpus.export("./data/youtube.jsonl")

## 2. Youtube Channel 

In [ ]:
# With config options passed as kwargs
corpus = si.gather(
    "youtube",
    ["https://www.youtube.com/@iamtrask"],
    playlistend=3,  # Cap at 3 videos to keep it fast
)

print(f"Videos found: {len(corpus.all_items())}")
for item in corpus.all_items():
    print(f"  [{item.published_at}] {item.title}")

In [ ]:
corpus.export("./data/youtube2.jsonl")

## 3. Facebook scraping

Requires `BRIGHTDATA_API_TOKEN`. Triggers a real scrape job and polls until done (~60-120s).

In [2]:
import os

token = os.getenv("BRIGHTDATA_API_TOKEN")
print(
    f"BRIGHTDATA_API_TOKEN : {token[:3] + '...' + token[-3:] if token else 'NOT SET -- BrightData cells will fail'}"
)

BRIGHTDATA_API_TOKEN : 7a5...1d5


In [ ]:
corpus = await si.async_gather(
    "facebook",
    ["https://www.facebook.com/profile.php?id=61583734012155"],
    author="Syft Influencer Test",
)

print(f"Items fetched: {len(corpus.all_items())}")
print(f"Author: {corpus.person}")

if corpus.all_items():
    item = corpus.all_items()[0]
    print("\nFirst item:")
    print(f"  Type: {type(item).__name__}")
    print(f"  Title: {item.title}")
    print(f"  Text: {(item.text or '')[:120]}")

In [ ]:
corpus.export("data/fb.jsonl")

## 4. Instagram scraping with posts_limit for testing

In [ ]:
corpus = await si.async_gather(
    "instagram",
    ["https://www.instagram.com/syft_influencer_test/"],
    author="Syft Influencer Test",
    timeout=120,
    poll_interval=5,
)

print(f"Items fetched: {len(corpus.all_items())}")
for i, item in enumerate(corpus.all_items()[:3], 1):
    print(f"\n{i}. {type(item).__name__}: {item.title}")

2026-04-14 04:37:14.343 | DEBUG    | syft_ingest.sources.youtube:__init__:83 - YtDlpFetcher initialized with config: {'socket_timeout': 30, 'playlistend': 50, 'download_full_video': False}
2026-04-14 04:37:14.343 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher YtDlpFetcher for youtube/yt-dlp
2026-04-14 04:37:14.343 | DEBUG    | syft_ingest.setup:register_fetchers:28 - Registered YtDlpFetcher for YouTube
2026-04-14 04:37:14.343 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher BrightDataFetcher for facebook/brightdata
2026-04-14 04:37:14.344 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher BrightDataFetcher for instagram/brightdata
2026-04-14 04:37:14.344 | DEBUG    | syft_ingest.setup:register_fetchers:37 - Registered BrightDataFetcher for Facebook and Instagram
2026-04-14 04:37:14.344 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher LocalFetcher for local/local
2

Items fetched: 24

1. ContentItem: 3856016616921100481

2. ContentItem: 3856009687577940838

3. ContentItem: 3856005579987608607


In [4]:
corpus.export("data/ig.jsonl")

2026-04-14 04:38:52.285 | INFO     | syft_ingest.core.exporters:_export_jsonl:76 - Exported 24 items to data/ig.jsonl


## 5. Concurrent fetching — async power

Run YouTube, Facebook, and Instagram **simultaneously**. Total time = slowest scrape, not the sum.

In [5]:
import asyncio
import time

start = time.time()

# All three run concurrently — event loop drives all poll loops simultaneously
corpus_yt, corpus_fb, corpus_ig = await asyncio.gather(
    si.async_gather("youtube", ["https://www.youtube.com/@iamtrask"], playlistend=3),
    si.async_gather(
        "facebook",
        ["https://www.facebook.com/profile.php?id=61583734012155"],
        author="Syft Influencer Test",
        posts_limit=5,
        timeout=300,
    ),
    si.async_gather(
        "instagram",
        ["https://www.instagram.com/syft_influencer_test/"],
        author="Syft Influencer Test",
        posts_limit=5,
        timeout=300,
    ),
)

elapsed = time.time() - start

print(f"YouTube:   {len(corpus_yt.all_items())} items")
print(f"Facebook:  {len(corpus_fb.all_items())} items")
print(f"Instagram: {len(corpus_ig.all_items())} items")
print(f"\nTotal time: {elapsed:.1f}s (sequential would be ~{elapsed * 2:.0f}s+)")

2026-04-14 04:41:29.632 | INFO     | syft_ingest.sources.brightdata:fetch_async:113 - Fetching 1 URL(s) for facebook
2026-04-14 04:41:29.633 | INFO     | syft_ingest.sources.youtube:fetch:124 - Fetching 1 YouTube URL(s)
2026-04-14 04:41:29.633 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:127 - Using platform scraper: facebook
2026-04-14 04:41:29.633 | INFO     | syft_ingest.sources.youtube:fetch:135 - Detected channel/playlist URL: https://www.youtube.com/@iamtrask
2026-04-14 04:41:29.634 | INFO     | syft_ingest.sources.brightdata:fetch_async:113 - Fetching 1 URL(s) for instagram
2026-04-14 04:41:29.634 | DEBUG    | syft_ingest.sources.youtube:_enumerate_channel:314 - Enumerating channel with limit=3
2026-04-14 04:41:29.635 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:127 - Using platform scraper: instagram
2026-04-14 04:41:30.071 | INFO     | syft_ingest.sources.brightdata:fetch_async:157 - Triggering facebook scrape for https://www.facebook.com/profile.php?id=6

YouTube:   3 items
Facebook:  5 items
Instagram: 5 items

Total time: 119.8s (sequential would be ~240s+)


In [6]:
corpus_yt.export("data/youtube_async.jsonl")
corpus_fb.export("data/facebook_async.jsonl")
corpus_ig.export("data/instagram_async.jsonl")

2026-04-14 04:43:54.329 | INFO     | syft_ingest.core.exporters:_export_jsonl:76 - Exported 3 items to data/youtube_async.jsonl
2026-04-14 04:43:54.330 | INFO     | syft_ingest.core.exporters:_export_jsonl:76 - Exported 5 items to data/facebook_async.jsonl
2026-04-14 04:43:54.331 | INFO     | syft_ingest.core.exporters:_export_jsonl:76 - Exported 5 items to data/instagram_async.jsonl


## 7. Error handling and validation

In [8]:
# Invalid platform raises ValueError
try:
    corpus = si.gather("tiktok", ["https://tiktok.com/user"])
except KeyError as e:
    print(f"✅ ValueError caught for unsupported platform: {e}")

# Missing URLs raises ValueError
try:
    corpus = si.gather("youtube", None)
except ValueError as e:
    print(f"✅ ValueError caught for missing URLs: {e}")

2026-04-14 04:44:25.978 | ERROR    | syft_ingest.core.gather:gather:83 - No fetcher registered for tiktok: "No fetcher registered for platform 'tiktok' and extractor 'brightdata'. Register one with register_fetcher()."
2026-04-14 04:44:25.978 | ERROR    | syft_ingest.core.gather:gather:80 - Invalid platform: Platform 'youtube' requires urls list


✅ ValueError caught for unsupported platform: "No fetcher registered for platform 'tiktok' and extractor 'brightdata'. Register one with register_fetcher()."
✅ ValueError caught for missing URLs: Platform 'youtube' requires urls list
